# Forecasts
This notebook defines the `Base Hierarchies`, imports the `Web Traffic` dataset, preprocesses it, then makes forecasts at each granularity level in each hierarchy, then reconciles these forecasts with `shrinkage` method. Of these forecasts, error metrics are calculated, which are saved to a csv.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

## Import and preprocess dataset

In [ ]:
X_raw = utils.fetch_dataset_web_traffic()
X = utils.filter_last_days_available(X_raw, 28*100)
print(f'{len(X.columns)} time series remain which have sufficient data')

Only keeping a subset of time series, and trimming them to end on a Sunday and have data points equal a multiple of 28 

In [ ]:
X = utils.reduce_dataset(X, 13000)
X = utils.align_fill_and_trim_series(X)

A look at the dataset on log scale

In [ ]:
plots.plot_heatmap(utils.reduce_dataset(X, 1000), True)

## Defining hierarchies and forecasting

The following hierarchies and their corresponding aggregation levels are defined as `Base Hierarchies`: 
- Minimal: [MONTHS, DAYS]
- Minimal+Weeks: [MONTHS, WEEKS, DAYS]
- Weeks+Biweeks: [MONTHS, FORTNIGHTS, WEEKS, DAYS]
- Weekends+Weekdays: [MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS]
- Semantic: [MONTHS, FORTNIGHTS, WEEKS, WEEKENDS, WEEKDAYS, DAYS]

In [ ]:
sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

A look at a summing matrix.

In [ ]:
plots.plot_summing_matrix(wweth.make_summing_matrix())

We are forecasting a for a horizon of 3 times 28 days, so roughly three months. This ensures that the horizon isn't so great on a daily level, but also not singular on monthly level.

Note that because base forecasts are independently made at each granularity and for each time series, I am making the forecasts only once, then copying the data to the other hierarchies (which all have a subset of the levels defined in `SemanticHierarchy`)

In [ ]:
h = 28 * 3

sth.resample_time_series(X).make_base_forecasts(h)
mth.copy_data(sth)
wth.copy_data(sth)
wweth.copy_data(sth)
wfth.copy_data(sth)

Reconciling the forecasts in each hierarcy independently, with the specified, `shrinkage` method, as we only have a limited number of residuals and vast amounts of hierarchy levels in a temporal hierarchical situation (i.e. the daily level becomes 28 independent levels)

In [ ]:
method = 'shr'

ForecastReconciliation(sth).reconcile(method)
ForecastReconciliation(mth).reconcile(method)
ForecastReconciliation(wth).reconcile(method)
ForecastReconciliation(wweth).reconcile(method)
ForecastReconciliation(wfth).reconcile(method)

## Calculate error metrics

Calculate the predefined error metrics in daily and monthly granularity in all the hierarchies, and saving them. 

In [ ]:
em = MergedErrorMetrics([
    ErrorMetrics(sth),
    ErrorMetrics(mth),
    ErrorMetrics(wth),
    ErrorMetrics(wweth),
    ErrorMetrics(wfth)
]).calc_error_metrics([DAYS, MONTHS])

em.save('metrics/base_hier_errors.csv')

A look at the metrics calculated

In [ ]:
em.metrics

### Calculate inconsistency error

In [ ]:
mth.resample_time_series(X).make_base_forecasts(28 * 3)

In [ ]:
inc = ForecastReconciliation(mth).inconcistency_errors(top_level=MONTHS, leaf_level=DAYS)
inc.to_csv('metrics/base_hier_inconsistency.csv')